# Probability of Default with Logistic Regression

**Family:** Credit Risk | **Level:** Beginner to Intermediate | **Dataset:** `retail_loans_dev` | **Data kind:** synthetic | **Module status:** Complete

Estimate 12 month PD, measure ranking and calibration, and compare with the true generating process.

> Data are synthetic. Numbers describe the generator, not any real bank.

## Table of Contents

1. [Problem Statement](#1-problem-statement)
2. [Learning Objectives](#2-learning-objectives)
3. [Fundamental Concepts](#3-fundamental-concepts)
4. [Why This Method?](#4-why-this-method)
5. [Research Foundation](#5-research-foundation)
6. [Mathematical Derivation](#6-mathematical-derivation)
7. [Worked Numerical Example](#7-worked-numerical-example)
8. [Dataset Exploration](#8-dataset-exploration)
9. [Data Preparation](#9-data-preparation)
10. [Implementation from Scratch](#10-implementation-from-scratch)
11. [Implementation using Libraries](#11-implementation-using-libraries)
12. [Model Evaluation](#12-model-evaluation)
13. [Visualization](#13-visualization)
14. [Interpretation](#14-interpretation)
15. [Experiments](#15-experiments)
16. [Results and Comparisons](#16-results-and-comparisons)
17. [Limitations and Model Risk](#17-limitations-and-model-risk)
18. [Practical Banking Application](#18-practical-banking-application)
19. [Exercises](#19-exercises)
20. [References](#20-references)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from risk_lab.reporting.results import find_root, save_config, save_result
from risk_lab.visualization.style import COLORS, apply_style, save_figure

apply_style()
pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 20)
MODULE = "probability_of_default"
ROOT = find_root()
SEED = 42
from risk_lab.credit.logistic import fit_logistic, predict_proba, sigmoid, standard_errors
from risk_lab.credit.metrics import (
    brier_score, calibration_table, gini, ks_statistic, log_loss, roc_auc,
)
from risk_lab.data.synthetic import (
    RETAIL_TRUE_INTERCEPT, RETAIL_TRUE_SLOPES, standardise_features, true_default_probability,
)

## 1. Problem Statement

A retail lender receives a loan application. It must decide whether to lend, at what price, and how much
capital and provision to hold. All three decisions need one number: **the probability that this borrower
defaults within 12 months (PD)**.

This notebook estimates PD with logistic regression on a synthetic loan sample. Because the sample comes from a
known generating process, the notebook can do something real data never allows: compare the estimated model with
the true probabilities and the true coefficients.

**Decision this supports:** whether a PD model is good enough to rank borrowers, to price risk, and to feed
provisioning, and which of those three uses the evidence supports.

## 2. Learning Objectives

After this notebook you can:

1. Write the logistic regression likelihood, its gradient and Hessian, and take one Newton step by hand.
2. Fit the model with a NumPy implementation, scikit-learn and statsmodels, and show the three agree.
3. Measure discrimination (AUC, Gini, KS), calibration (Brier, deciles) and compare with the best possible model.
4. Read coefficients as odds ratios and compare them with known true values.
5. Measure how sample size and regularisation change accuracy.
6. Turn PD into risk grades and an approval cutoff, and state what the numbers do not prove.

## 3. Fundamental Concepts

| Term | Meaning |
|---|---|
| Default | Event defined by the dataset: 1 if the loan defaults within 12 months of origination |
| PD | Probability of default over a horizon. Here 12 months |
| Odds | p / (1 - p). Logistic regression is linear in the log odds |
| Odds ratio | exp(coefficient). Multiplier on the odds for a one unit rise in the feature |
| Discrimination | Ability to rank bad borrowers above good ones. AUC, Gini, KS |
| Calibration | Whether predicted PD matches observed default rate. A model can rank well and still be miscalibrated |
| Gini | 2 * AUC - 1 |
| KS | Largest gap between cumulative bad and good distributions across the score |

Discrimination and calibration are different properties. Ranking is enough for an approve or decline cutoff.
Provisioning and pricing need calibrated probabilities.

## 4. Why This Method?

**Why logistic regression for PD?**

| Option | Strength | Weakness |
|---|---|---|
| Logistic regression | Output is a probability. Coefficients are readable. Easy to document and validate | Linear in the log odds unless features are transformed |
| Gradient boosting or neural network | Can capture interactions | Harder to explain and validate. Benchmarks of credit scoring methods report that gains over well built simpler models depend on the data and are not guaranteed (see Section 5) |
| Single rule or expert grade | Cheap | No measured accuracy |

Logistic regression is the baseline every other method must beat. Acceptance criteria used in this notebook,
on a test set not used for fitting:

1. AUC at least 0.70 (a teaching threshold, not a regulatory one).
2. Test AUC within 0.02 of the AUC of the true generating probability.
3. Mean predicted PD within 10 percent (relative) of the observed default rate.

## 5. Research Foundation

### Academic and book sources

- **Benchmarking state-of-the-art classification algorithms for credit scoring: An update of research** (Lessmann, S., Baesens, B., Seow, H.-V. and Thomas, L. C., 2015). Large benchmark of 41 classifiers on eight real credit scoring data sets, with several accuracy indicators and a financial impact view. [verified: RePEc IDEAS, dblp and library records, web search 2026-11-25]
- **Deep learning for credit scoring: Do or don't?** (Gunnarsson, B. R., vanden Broucke, S., Baesens, B., Óskarsdóttir, M. and Lemahieu, W., 2021). Compares deep learning with conventional and ensemble credit scoring methods. The authors conclude deep learning looks inappropriate there and that XGBoost is preferred when classification performance is the main goal. [verified: RePEc IDEAS, dblp and library records, web search 2026-11-25]

### Regulatory and standard setter sources

- **Basel III: Finalising post-crisis reforms (BCBS d424)** (Basel Committee on Banking Supervision, 2017). Status: integrated into the consolidated Basel Framework. Finalises the Basel III reforms, including the revised standardised approach for credit risk. [verified: BIS page, web search 2026-11-25]
- **SR 26-2: Revised Guidance on Model Risk Management** (Federal Reserve, OCC and FDIC, 2026). Status: current. Current US model risk guidance with a risk based approach. The letter says it is most relevant to organisations above 30 billion dollars in assets and states no effective date. [verified: Federal Reserve page, fetched 2026-11-25]

**How this notebook uses them.** Lessmann et al. and Gunnarsson et al. are used only to frame the choice of logistic regression as the baseline for credit scoring. Their experiments use real credit datasets and are not reproduced here. SR 26-2 is cited for the model risk language in the limitations. The Basel entry is context for how PD feeds capital, which this notebook does not compute.

**Academic findings versus regulatory requirements.** Papers describe what methods can do. Regulations and accounting standards say what a supervised institution must do. This notebook labels which is which, and it does not reproduce any paper's experiment. Data differ, so numbers are not comparable with published results.

## 6. Mathematical Derivation

Observation $i$ has features $x_i$ and outcome $y_i \in \{0,1\}$. The model is

$$p_i = \Pr(y_i = 1 \mid x_i) = \frac{1}{1 + e^{-(\beta_0 + x_i^\top \beta)}}$$

The log likelihood is

$$\ell(\beta) = \sum_{i=1}^{n} \left[ y_i \ln p_i + (1 - y_i)\ln(1 - p_i) \right]$$

Gradient and Hessian, with $X$ the design matrix with a column of ones and $W = \mathrm{diag}\big(p_i(1-p_i)\big)$:

$$\nabla \ell = X^\top (y - p), \qquad H = -X^\top W X$$

Newton Raphson update:

$$\beta^{new} = \beta + (X^\top W X)^{-1} X^\top (y - p)$$

The inverse of $X^\top W X$ at the solution is the large sample covariance matrix of the estimates, so
standard errors are the square roots of its diagonal.

Interpretation: $e^{\beta_j}$ is the factor by which the **odds** of default change for a one unit rise in feature $j$.

## 7. Worked Numerical Example

**Part A. One Newton step on eight observations.** Start from all coefficients at zero, so every predicted
probability is 0.5.

In [2]:
x = np.array([-2.0, -1.0, -1.0, 0.0, 0.0, 1.0, 1.0, 2.0])
y = np.array([0, 0, 1, 0, 1, 0, 1, 1])
design = np.column_stack([np.ones(len(x)), x])
beta = np.zeros(2)
p = sigmoid(design @ beta)
gradient = design.T @ (y - p)
weights = p * (1 - p)
hessian_neg = design.T @ (design * weights[:, None])
step = np.linalg.solve(hessian_neg, gradient)
print("probabilities at start:", p)
print("gradient:", gradient)
print("information matrix:\n", hessian_neg)
print("step:", step)
print("beta after one step:", beta + step)

probabilities at start: [0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5]
gradient: [0. 2.]
information matrix:
 [[2. 0.]
 [0. 3.]]
step: [0.         0.66666667]
beta after one step: [0.         0.66666667]


In [3]:
# full fit, same data
tiny = fit_logistic(x.reshape(-1, 1), y)
print("converged intercept:", round(tiny["intercept"], 6), "slope:", round(float(tiny["slopes"][0]), 6))
print("iterations:", tiny["iterations"])
assert tiny["iterations"] < 15

converged intercept: -0.0 slope: 0.756308
iterations: 5


**Part B. Reading a coefficient.** Take a borrower whose debt to income ratio is one standard deviation above
average and who is average on every other feature. The true model gives the logit below.

In [4]:
logit = RETAIL_TRUE_INTERCEPT + RETAIL_TRUE_SLOPES["dti_z"] * 1.0
print("logit:", round(logit, 4))
print("PD:", round(float(sigmoid(logit)), 4))
print("PD of an average borrower:", round(float(sigmoid(RETAIL_TRUE_INTERCEPT)), 4))
print("odds ratio for one sd of dti:", round(float(np.exp(RETAIL_TRUE_SLOPES["dti_z"])), 4))

logit: -2.55
PD: 0.0724
PD of an average borrower: 0.0431
odds ratio for one sd of dti: 1.7333


## 8. Dataset Exploration

In [5]:
loans = pd.read_csv(ROOT / "datasets" / "synthetic" / "retail_loans_dev.csv")
print("shape:", loans.shape)
print("default rate:", round(loans["default_12m"].mean(), 4))
print("missing values:", int(loans.isna().sum().sum()))
print(loans.describe().T[["mean", "std", "min", "50%", "max"]].round(3))

shape: (15000, 14)
default rate: 0.0637
missing values: 0
                           mean        std       min        50%         max
loan_id                7500.500   4330.271     1.000   7500.500   15000.000
age                      44.830     14.085    21.000     45.000      69.000
income                46930.951  27934.694  4730.000  40023.500  393636.000
employment_years          7.010      4.362     0.100      6.000      34.500
credit_history_years      9.447      5.461     0.500      8.400      48.700
dti                       0.287      0.154     0.010      0.265       0.871
utilization               0.383      0.195     0.004      0.365       0.977
ltv                       0.720      0.178     0.100      0.721       1.200
delinq_2y                 0.254      0.504     0.000      0.000       4.000
loan_amount           13093.779   8768.822  1050.000  10840.000  120030.000
term_months              38.331     15.015    12.000     36.000      60.000
interest_rate             0.07

In [6]:
by_region = loans.groupby("region")["default_12m"].agg(["count", "mean"]).round(4)
by_delinq = loans.groupby(loans["delinq_2y"].clip(upper=3))["default_12m"].agg(["count", "mean"]).round(4)
print(by_region)
print(by_delinq)

from scipy.stats import chi2_contingency
table = pd.crosstab(loans["region"], loans["default_12m"])
print("region chi square p value:", round(chi2_contingency(table)[1], 4))

        count    mean
region               
east     3724  0.0607
north    4465  0.0632
south    3873  0.0622
west     2938  0.0705
           count    mean
delinq_2y               
0          11632  0.0444
1           2973  0.1086
2            355  0.2789
3             40  0.4250
region chi square p value: 0.3953


In [7]:
numeric = loans.drop(columns=["loan_id", "region"])
corr_target = numeric.corr()["default_12m"].drop("default_12m").sort_values()
print(corr_target.round(3))

income                 -0.086
credit_history_years   -0.057
term_months            -0.013
age                    -0.011
loan_amount            -0.009
employment_years       -0.002
ltv                     0.074
utilization             0.076
dti                     0.157
delinq_2y               0.179
interest_rate           0.196
Name: default_12m, dtype: float64


## 9. Data Preparation

Steps, in order:

1. Split 70 percent train and 30 percent test with a fixed seed. The test set is not used for any fitting choice.
2. Build two feature sets. **Model A** uses the seven features that drive default in the generator, scaled with the
   generator's reference scales. A real modeller would not know these. They are used here so coefficients can be
   compared with the truth. **Model B** uses every numeric column, scaled with train means and standard deviations,
   as a real modeller would.
3. Region is dropped from both sets. Its contribution is tested in section 8 and is expected to be zero.

In [8]:
rng = np.random.default_rng(SEED)
order = rng.permutation(len(loans))
n_train = int(0.7 * len(loans))
train = loans.iloc[order[:n_train]].reset_index(drop=True)
test = loans.iloc[order[n_train:]].reset_index(drop=True)
print("train rows:", len(train), "defaults:", int(train["default_12m"].sum()))
print("test rows:", len(test), "defaults:", int(test["default_12m"].sum()))

# model A features
xa_train = standardise_features(train)
xa_test = standardise_features(test)
names_a = list(xa_train.columns)

# model B features
def make_b(frame):
    out = frame[["age", "employment_years", "credit_history_years", "dti", "utilization", "ltv",
                 "delinq_2y", "term_months", "interest_rate"]].copy()
    out["log_income"] = np.log(frame["income"])
    out["log_loan_amount"] = np.log(frame["loan_amount"])
    return out

xb_train_raw = make_b(train)
xb_test_raw = make_b(test)
mean_b = xb_train_raw.mean()
std_b = xb_train_raw.std()
xb_train = (xb_train_raw - mean_b) / std_b
xb_test = (xb_test_raw - mean_b) / std_b
names_b = list(xb_train.columns)
y_train = train["default_12m"].to_numpy()
y_test = test["default_12m"].to_numpy()
print("model A features:", names_a)
print("model B features:", names_b)

train rows: 10500 defaults: 676
test rows: 4500 defaults: 280
model A features: ['dti_z', 'utilization_z', 'ltv_z', 'log_income_z', 'credit_history_years_z', 'delinq_2y_z', 'interest_rate_z']
model B features: ['age', 'employment_years', 'credit_history_years', 'dti', 'utilization', 'ltv', 'delinq_2y', 'term_months', 'interest_rate', 'log_income', 'log_loan_amount']


## 10. Implementation from Scratch

The NumPy implementation in `risk_lab.credit.logistic` follows section 6 exactly: Newton Raphson, no optimiser library.

In [9]:
fit_a = fit_logistic(xa_train.to_numpy(), y_train)
fit_b = fit_logistic(xb_train.to_numpy(), y_train)
se_a = standard_errors(xa_train.to_numpy(), fit_a["intercept"], fit_a["slopes"])
print("iterations A:", fit_a["iterations"], "iterations B:", fit_b["iterations"])

compare = pd.DataFrame(
    {
        "true": [RETAIL_TRUE_INTERCEPT] + [RETAIL_TRUE_SLOPES[n] for n in names_a],
        "estimate": [fit_a["intercept"]] + list(fit_a["slopes"]),
        "std_error": list(se_a),
    },
    index=["intercept"] + names_a,
)
compare["error"] = compare["estimate"] - compare["true"]
compare["abs_error_in_se"] = compare["error"].abs() / compare["std_error"]
print(compare.round(4))

iterations A: 8 iterations B: 8
                        true  estimate  std_error   error  abs_error_in_se
intercept              -3.10   -3.2142     0.0893 -0.1142           1.2777
dti_z                   0.55    0.6009     0.0610  0.0509           0.8354
utilization_z           0.40    0.3222     0.0424 -0.0778           1.8350
ltv_z                   0.30    0.3585     0.0434  0.0585           1.3476
log_income_z           -0.45   -0.4518     0.0426 -0.0018           0.0431
credit_history_years_z -0.25   -0.3371     0.0541 -0.0871           1.6096
delinq_2y_z             0.50    0.5193     0.0476  0.0193           0.4060
interest_rate_z         0.20    0.1823     0.1415 -0.0177           0.1252


## 11. Implementation using Libraries

Two library fits on the same data. They must agree with the NumPy fit. Any gap points to a bug or to a
different optimiser tolerance.

In [10]:
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression

sm_fit = sm.Logit(y_train, sm.add_constant(xa_train.to_numpy())).fit(disp=0)
sk_fit = LogisticRegression(C=np.inf, max_iter=1000, tol=1e-10).fit(xa_train.to_numpy(), y_train)

mine = np.concatenate([[fit_a["intercept"]], fit_a["slopes"]])
gap_sm = np.max(np.abs(sm_fit.params - mine))
gap_sk = np.max(np.abs(np.concatenate([sk_fit.intercept_, sk_fit.coef_[0]]) - mine))
gap_se = np.max(np.abs(sm_fit.bse - se_a))
print("max coefficient gap vs statsmodels:", f"{gap_sm:.2e}")
print("max coefficient gap vs scikit-learn:", f"{gap_sk:.2e}")
print("max standard error gap vs statsmodels:", f"{gap_se:.2e}")
assert gap_sm < 1e-5
assert gap_sk < 1e-3
assert gap_se < 1e-5

max coefficient gap vs statsmodels: 8.33e-17
max coefficient gap vs scikit-learn: 1.68e-08
max standard error gap vs statsmodels: 8.33e-17


In [11]:
# model B significance
sm_b = sm.Logit(y_train, sm.add_constant(xb_train.to_numpy())).fit(disp=0)
table_b = pd.DataFrame(
    {"coef": sm_b.params[1:], "p_value": sm_b.pvalues[1:]}, index=names_b
).sort_values("p_value")
print(table_b.round(4))

                        coef  p_value
delinq_2y             0.5267   0.0000
log_income           -0.4550   0.0000
dti                   0.6199   0.0000
ltv                   0.3558   0.0000
utilization           0.3140   0.0000
credit_history_years -0.3004   0.0000
interest_rate         0.0966   0.1925
employment_years      0.0429   0.3181
log_loan_amount      -0.0269   0.5251
age                  -0.0255   0.5538
term_months          -0.0010   0.9806


## 12. Model Evaluation

Everything below uses the **test** set. Baselines:

* **Constant PD**: every borrower gets the train default rate. Zero discrimination by construction.
* **Oracle**: the true generating probability. No model fitted on this data can beat it on average, so it is the ceiling.
* **Single feature**: logistic regression on debt to income alone.

In [12]:
p_a = predict_proba(xa_test.to_numpy(), fit_a["intercept"], fit_a["slopes"])
p_b = predict_proba(xb_test.to_numpy(), fit_b["intercept"], fit_b["slopes"])
oracle = true_default_probability(test).to_numpy()
base_rate = float(y_train.mean())
p_const = np.full(len(test), base_rate)

dti_fit = fit_logistic(xa_train[["dti_z"]].to_numpy(), y_train)
p_dti = predict_proba(xa_test[["dti_z"]].to_numpy(), dti_fit["intercept"], dti_fit["slopes"])

candidates = {
    "constant": p_const,
    "dti only": p_dti,
    "model B (all numeric)": p_b,
    "model A (7 drivers)": p_a,
    "oracle (true PD)": oracle,
}
rows = []
for name, prob in candidates.items():
    auc = roc_auc(y_test, prob) if np.ptp(prob) > 0 else 0.5
    rows.append(
        {
            "model": name,
            "auc": auc,
            "gini": 2 * auc - 1,
            "ks": ks_statistic(y_test, prob) if np.ptp(prob) > 0 else 0.0,
            "brier": brier_score(y_test, prob),
            "log_loss": log_loss(y_test, prob),
            "mean_pd": float(np.mean(prob)),
        }
    )
evaluation = pd.DataFrame(rows).set_index("model")
print(evaluation.round(4))
print("observed test default rate:", round(float(y_test.mean()), 4))

                          auc    gini      ks   brier  log_loss  mean_pd
model                                                                   
constant               0.5000  0.0000  0.0000  0.0584    0.2331   0.0644
dti only               0.6540  0.3080  0.2318  0.0573    0.2244   0.0636
model B (all numeric)  0.7848  0.5695  0.4683  0.0532    0.1999   0.0633
model A (7 drivers)    0.7855  0.5710  0.4725  0.0532    0.1996   0.0632
oracle (true PD)       0.7853  0.5707  0.4538  0.0530    0.1992   0.0659
observed test default rate: 0.0622


In [13]:
auc_a = float(evaluation.loc["model A (7 drivers)", "auc"])
auc_oracle = float(evaluation.loc["oracle (true PD)", "auc"])
mean_gap_pct = 100 * (float(np.mean(p_a)) - float(y_test.mean())) / float(y_test.mean())
criteria = {
    "auc_at_least_0.70": auc_a >= 0.70,
    "auc_within_0.02_of_oracle": (auc_oracle - auc_a) <= 0.02,
    "mean_pd_within_10pct_of_observed": abs(mean_gap_pct) <= 10,
}
print("mean PD gap (percent of observed rate):", round(mean_gap_pct, 2))
print(criteria)

mean PD gap (percent of observed rate): 1.53
{'auc_at_least_0.70': True, 'auc_within_0.02_of_oracle': True, 'mean_pd_within_10pct_of_observed': True}


## 13. Visualization

In [14]:
cal = pd.DataFrame(calibration_table(y_test, p_a, n_bins=10))
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

axes[0].plot([0, cal["mean_predicted"].max() * 1.1], [0, cal["mean_predicted"].max() * 1.1], color=COLORS["grey"], linestyle=":")
axes[0].plot(cal["mean_predicted"], cal["observed_rate"], marker="o", color=COLORS["main"])
axes[0].set_xlabel("mean predicted PD")
axes[0].set_ylabel("observed default rate")
axes[0].set_title("Calibration by decile, model A")

order_a = np.argsort(-p_a)
cum_bad = np.cumsum(y_test[order_a]) / y_test.sum()
cum_good = np.cumsum(1 - y_test[order_a]) / (1 - y_test).sum()
axes[1].plot(cum_good, cum_bad, color=COLORS["main"], label="model A")
order_o = np.argsort(-oracle)
axes[1].plot(np.cumsum(1 - y_test[order_o]) / (1 - y_test).sum(), np.cumsum(y_test[order_o]) / y_test.sum(), color=COLORS["third"], linestyle="--", label="oracle")
axes[1].plot([0, 1], [0, 1], color=COLORS["grey"], linestyle=":")
axes[1].set_xlabel("cumulative share of goods")
axes[1].set_ylabel("cumulative share of bads")
axes[1].set_title("ROC curve")
axes[1].legend()

axes[2].hist(p_a[y_test == 0], bins=40, alpha=0.6, color=COLORS["main"], label="good", density=True)
axes[2].hist(p_a[y_test == 1], bins=40, alpha=0.6, color=COLORS["second"], label="bad", density=True)
axes[2].set_xlabel("predicted PD")
axes[2].set_title("Score distribution")
axes[2].legend()
save_figure(fig, MODULE, "pd_evaluation.png")
plt.show()
print(cal.round(4).to_string(index=False))

 bin  count  mean_predicted  observed_rate
   1    450          0.0063         0.0067
   2    450          0.0118         0.0156
   3    450          0.0172         0.0200
   4    450          0.0233         0.0222
   5    450          0.0304         0.0178
   6    450          0.0397         0.0422
   7    450          0.0532         0.0444
   8    450          0.0749         0.1044
   9    450          0.1144         0.1267
  10    450          0.2606         0.2222


## 14. Interpretation

How to read the output of the next cell:

* **Coefficients** within about two standard errors of the truth show the estimator is behaving. Some misses are expected by chance: with a 95 percent interval, about one coefficient in twenty can miss.
* **Model B** contains features that have no true effect. Their p values should be spread over 0 to 1, not clustered near 0.
* **Model A against the oracle**: model A has the right functional form by construction, so any gap is the price of estimating coefficients from a finite sample. On a finite test set the gap can even be slightly negative, because the test labels are random draws and the oracle is only best on average, not on every sample.

In [15]:
inside = int((compare["abs_error_in_se"].iloc[1:] <= 1.96).sum())
print("slopes within 1.96 standard errors of truth:", inside, "of", len(names_a))
noise = ["age", "employment_years", "log_loan_amount", "term_months"]
print("p values of features with zero true effect:")
print(table_b.loc[noise, "p_value"].round(3))
print("AUC gap to oracle:", round(auc_oracle - auc_a, 4))
print("Brier gap to oracle:", round(float(evaluation.loc["model A (7 drivers)", "brier"] - evaluation.loc["oracle (true PD)", "brier"]), 5))

slopes within 1.96 standard errors of truth: 7 of 7
p values of features with zero true effect:
age                 0.554
employment_years    0.318
log_loan_amount     0.525
term_months         0.981
Name: p_value, dtype: float64
AUC gap to oracle: -0.0002
Brier gap to oracle: 0.00022


## 15. Experiments

**Experiment A: sample size.** Draw random subsets of the train set at four sizes, 30 times each. Fit model A.
Record test AUC and the root mean squared error (RMSE) of the seven slopes against the truth.

**Experiment B: ridge penalty.** At 1000 training rows, repeat with L2 penalties 0, 10, 100 and 1000 on the slopes.

In [16]:
true_slopes = np.array([RETAIL_TRUE_SLOPES[n] for n in names_a])
sizes = [300, 1000, 3000, len(train)]
repeat = 30
rng_exp = np.random.default_rng(SEED + 1)
rows = []
for size in sizes:
    reps = 1 if size == len(train) else repeat
    for rep in range(reps):
        pick = rng_exp.choice(len(train), size=size, replace=False)
        fit = fit_logistic(xa_train.to_numpy()[pick], y_train[pick])
        prob = predict_proba(xa_test.to_numpy(), fit["intercept"], fit["slopes"])
        rows.append(
            {
                "n_train": size,
                "auc": roc_auc(y_test, prob),
                "slope_rmse": float(np.sqrt(np.mean((fit["slopes"] - true_slopes) ** 2))),
            }
        )
size_table = pd.DataFrame(rows).groupby("n_train").agg(
    auc_mean=("auc", "mean"), auc_sd=("auc", "std"), slope_rmse=("slope_rmse", "mean")
)
print(size_table.round(4))

         auc_mean  auc_sd  slope_rmse
n_train                              
300        0.7503  0.0305      0.4372
1000       0.7760  0.0068      0.2186
3000       0.7832  0.0023      0.1254
10500      0.7855     NaN      0.0539


In [17]:
rows = []
for penalty in [0, 10, 100, 1000]:
    for rep in range(repeat):
        pick = rng_exp.choice(len(train), size=1000, replace=False)
        fit = fit_logistic(xa_train.to_numpy()[pick], y_train[pick], l2=penalty)
        prob = predict_proba(xa_test.to_numpy(), fit["intercept"], fit["slopes"])
        rows.append(
            {
                "l2": penalty,
                "auc": roc_auc(y_test, prob),
                "brier": brier_score(y_test, prob),
                "slope_rmse": float(np.sqrt(np.mean((fit["slopes"] - true_slopes) ** 2))),
            }
        )
ridge_table = pd.DataFrame(rows).groupby("l2").mean()
print(ridge_table.round(4))

         auc   brier  slope_rmse
l2                              
0     0.7761  0.0541      0.2270
10    0.7765  0.0537      0.1208
100   0.7709  0.0551      0.2462
1000  0.7654  0.0578      0.3718


In [18]:
fig, ax = plt.subplots()
ax.errorbar(size_table.index, size_table["auc_mean"], yerr=size_table["auc_sd"].fillna(0), marker="o", color=COLORS["main"], label="test AUC, model A")
ax.axhline(auc_oracle, color=COLORS["third"], linestyle="--", label="oracle AUC")
ax.set_xscale("log")
ax.set_xlabel("training rows")
ax.set_ylabel("test AUC")
ax.set_title("Accuracy against sample size")
ax.legend()
save_figure(fig, MODULE, "auc_by_sample_size.png")
plt.show()

In [19]:
print("slope RMSE falls as n grows:", bool(size_table["slope_rmse"].is_monotonic_decreasing))
print("l2 with lowest slope RMSE at n=1000:", int(ridge_table["slope_rmse"].idxmin()))

slope RMSE falls as n grows: True
l2 with lowest slope RMSE at n=1000: 10


## 16. Results and Comparisons

In [20]:
summary = evaluation[["auc", "gini", "ks", "brier", "log_loss"]].round(4)
print(summary)
print()
print(criteria)
metrics = {
    "test_auc": evaluation["auc"].to_dict(),
    "test_gini": evaluation["gini"].to_dict(),
    "test_ks": evaluation["ks"].to_dict(),
    "test_brier": evaluation["brier"].to_dict(),
    "observed_test_default_rate": float(y_test.mean()),
    "mean_predicted_pd_model_a": float(np.mean(p_a)),
    "mean_pd_gap_pct_of_observed": mean_gap_pct,
    "newton_iterations_model_a": fit_a["iterations"],
    "coefficient_gap_vs_statsmodels": float(gap_sm),
    "slopes_within_1_96_se": inside,
    "acceptance_criteria": criteria,
    "sample_size_experiment": size_table.reset_index().to_dict(orient="records"),
    "ridge_experiment_n1000": ridge_table.reset_index().to_dict(orient="records"),
}
config = {"n_loans": len(loans), "train_share": 0.7, "seed": SEED, "features_model_a": names_a, "features_model_b": names_b}
save_config(MODULE, config)
path = save_result(
    MODULE,
    methodology="Logistic regression by Newton Raphson, checked against statsmodels and scikit-learn, evaluated on a held out test set",
    data_kind="synthetic",
    dataset="datasets/synthetic/retail_loans_dev.csv",
    n_obs=int(len(loans)),
    seed=SEED,
    parameters=config,
    assumptions=[
        "default follows a logistic model of seven standardised features",
        "region, age, employment_years, loan_amount and term_months have zero true effect",
        "no time dimension, one cross section",
        "acceptance thresholds are teaching choices",
    ],
    metrics=metrics,
    baseline={"constant_pd_auc": 0.5, "dti_only_auc": float(evaluation.loc["dti only", "auc"]), "oracle_auc": auc_oracle},
    limitations=[
        "the fitted model has the true functional form, so results are optimistic",
        "synthetic data, no real portfolio behaviour",
        "no out of time test in this module",
    ],
)
print("saved", path.name)

                          auc    gini      ks   brier  log_loss
model                                                          
constant               0.5000  0.0000  0.0000  0.0584    0.2331
dti only               0.6540  0.3080  0.2318  0.0573    0.2244
model B (all numeric)  0.7848  0.5695  0.4683  0.0532    0.1999
model A (7 drivers)    0.7855  0.5710  0.4725  0.0532    0.1996
oracle (true PD)       0.7853  0.5707  0.4538  0.0530    0.1992

{'auc_at_least_0.70': True, 'auc_within_0.02_of_oracle': True, 'mean_pd_within_10pct_of_observed': True}
saved probability_of_default.json


## 17. Limitations and Model Risk

**Limitations**

1. **Correct specification.** The data come from a logistic model and Model A uses the true features. Real data have no such luxury. Expect a larger gap between a fitted model and the best possible model.
2. **No time dimension.** Real PD models face changing economic conditions. Out of time performance is tested in `credit_scoring`, which uses a shifted sample.
3. **Class balance.** Default is the rare class. With few defaults, coefficient error rises sharply. Experiment A quantifies it here.
4. **Point in time versus through the cycle.** This PD is conditional on features at origination. Whether it behaves like a point in time or a cycle average PD depends on the features and is not tested here.
5. **Thresholds** (AUC 0.70, 0.02 gap, 10 percent) are teaching choices.

**Model risk.** Main risks are using a ranking tool for pricing without checking calibration, and extrapolating
beyond the feature ranges of the sample. Independent validation would repeat the fit, test stability and review
the default definition. This notebook is not an independent validation.

## 18. Practical Banking Application

**Risk grades.** Lenders seldom act on a raw probability. They map PD to a small number of grades, often called a
master scale, then attach pricing and limits to the grades. The grades below use test set PD quintiles.
This is a simplified teaching version. Real master scales fix PD boundaries in advance so grades keep their meaning
across time.

**Approval cutoff.** For each cutoff on PD, the table shows the approval rate, the bad rate among approved
loans and the bad rate among rejected loans.

In [21]:
grades = pd.qcut(pd.Series(p_a), q=5, labels=["A", "B", "C", "D", "E"])
grade_table = pd.DataFrame({"grade": grades, "pd": p_a, "default": y_test}).groupby("grade", observed=True).agg(
    loans=("default", "size"), mean_pd=("pd", "mean"), observed_rate=("default", "mean")
)
print(grade_table.round(4))

       loans  mean_pd  observed_rate
grade                               
A        900   0.0091         0.0111
B        900   0.0202         0.0211
C        900   0.0350         0.0300
D        900   0.0641         0.0744
E        900   0.1875         0.1744


In [22]:
rows = []
for quantile in [0.5, 0.7, 0.8, 0.9, 1.0]:
    cutoff = np.quantile(p_a, quantile)
    approved = p_a <= cutoff
    rows.append(
        {
            "approval_rate": float(approved.mean()),
            "cutoff_pd": float(cutoff),
            "bad_rate_approved": float(y_test[approved].mean()),
            "bad_rate_rejected": float(y_test[~approved].mean()) if (~approved).any() else np.nan,
        }
    )
print(pd.DataFrame(rows).round(4).to_string(index=False))

 approval_rate  cutoff_pd  bad_rate_approved  bad_rate_rejected
           0.5     0.0345             0.0164             0.1080
           0.7     0.0616             0.0241             0.1511
           0.8     0.0912             0.0342             0.1744
           0.9     0.1439             0.0444             0.2222
           1.0     0.7921             0.0622                NaN


## 19. Exercises

1. Add `region` as four dummy variables to model B and test whether any is significant. Explain what a significant result would mean for this dataset.
2. Re-run with seed 7 for the split. Report the range of test AUC across five seeds.
3. In Experiment A, find the training size where the mean AUC is within 0.01 of the oracle. State how many defaults that size contains.
4. Replace `interest_rate` with the ratio `loan_amount / income`. Does AUC change? Explain why interest rate has a true effect in this generator although it is itself driven by dti and delinquencies.
5. Argue for or against using this model to set provisions without recalibration. Use the calibration table.

## 20. References

Entries are grouped by type. A reference marked NOT VERIFIED must be checked before it is cited.

- **Lessmann, S., Baesens, B., Seow, H.-V. and Thomas, L. C. (2015).** Benchmarking state-of-the-art classification algorithms for credit scoring: An update of research. European Journal of Operational Research 247(1):124-136. https://doi.org/10.1016/j.ejor.2015.05.030
  - Type: academic. Status: current. Check: verified: RePEc IDEAS, dblp and library records, web search 2026-11-25
- **Gunnarsson, B. R., vanden Broucke, S., Baesens, B., Óskarsdóttir, M. and Lemahieu, W. (2021).** Deep learning for credit scoring: Do or don't?. European Journal of Operational Research 295(1):292-305. https://doi.org/10.1016/j.ejor.2021.03.006
  - Type: academic. Status: current. Check: verified: RePEc IDEAS, dblp and library records, web search 2026-11-25
- **Basel Committee on Banking Supervision (2017).** Basel III: Finalising post-crisis reforms (BCBS d424). Bank for International Settlements, 7 December 2017, consolidated into the Basel Framework. https://www.bis.org/bcbs/publ/d424.htm
  - Type: regulatory. Status: integrated into the consolidated Basel Framework. Check: verified: BIS page, web search 2026-11-25
- **Federal Reserve, OCC and FDIC (2026).** SR 26-2: Revised Guidance on Model Risk Management. 17 April 2026, supersedes SR 11-7 and SR 21-8. https://www.federalreserve.gov/supervisionreg/srletters/sr2602.htm
  - Type: regulatory. Status: current. Check: verified: Federal Reserve page, fetched 2026-11-25